In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("../data/cleaned/cleaned_credit_default.csv")

In [4]:
df.shape

(149016, 19)

In [5]:
df.columns.tolist()

['SeriousDlqin2yrs',
 'RevolvingUtilizationOfUnsecuredLines',
 'age',
 'NumberOfTime30-59DaysPastDueNotWorse',
 'DebtRatio',
 'MonthlyIncome',
 'NumberOfOpenCreditLinesAndLoans',
 'NumberOfTimes90DaysLate',
 'NumberRealEstateLoansOrLines',
 'NumberOfTime60-89DaysPastDueNotWorse',
 'NumberOfDependents',
 'TotalDelinquencies',
 'AgeGroup',
 'HighUtilization',
 'IncomeMissing',
 'TotalCreditLines',
 'Has90DaysLate',
 'HasAnyDelinquency',
 'IncomePerCreditLine']

In [6]:
y = df["SeriousDlqin2yrs"]
X = df.drop(columns=["SeriousDlqin2yrs"])

In [9]:
print("X shape:", X.shape)
print("y shape:", y.shape)
df.dtypes

X shape: (149016, 18)
y shape: (149016,)


SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64
TotalDelinquencies                        int64
AgeGroup                                 object
HighUtilization                           int64
IncomeMissing                             int64
TotalCreditLines                          int64
Has90DaysLate                             int64
HasAnyDelinquency                         int64
IncomePerCreditLine                     float64
dtype: object

In [13]:
X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify = y,
    random_state=42
)

In [14]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (119212, 18)
X_test: (29804, 18)
y_train: (119212,)
y_test: (29804,)


In [15]:
print("Train target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Train target distribution:
SeriousDlqin2yrs
0    0.933748
1    0.066252
Name: proportion, dtype: float64

Test target distribution:
SeriousDlqin2yrs
0    0.933767
1    0.066233
Name: proportion, dtype: float64


In [17]:
categorical_cols = ["AgeGroup"]
numeric_cols = X_train.columns.difference(categorical_cols).to_list()

In [19]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [20]:
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

In [21]:
print(X_train_cat.shape)
print(X_test_cat.shape)

(119212, 7)
(29804, 7)


In [22]:
X_train_num = X_train[numeric_cols].to_numpy()
X_test_num = X_test[numeric_cols].to_numpy()

In [33]:
X_train_final = np.hstack([ # Final train set after encoding category columns
    X_train_num,
    X_train_cat
])

X_test_final = np.hstack([   # Final test set after encoding category columns
    X_test_num,
    X_test_cat
])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_final) # Only scaled for linear Model (Logistic Regression)
X_test_scaled = scaler.transform(X_test_final)

In [34]:
print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)

X_train_final: (119212, 24)
X_test_final: (29804, 24)


# Training Baseline Model

### Logistic Regression

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train_scaled, y_train)

y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.9370554288015032
Precision: 0.5983935742971888
Recall   : 0.15096251266464034
F1 Score : 0.24110032362459546
ROC-AUC  : 0.8249825889637512


In [41]:
# Because the target is highly imbalanced, the default 0.5 threshold produces low recall. Lowering the threshold increases recall at the cost of precision. The optimal threshold should be selected based on the desired precision-recall trade-off, which we will investigate later.

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred = (y_prob >= threshold).astype(int)

    print(
        threshold,
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    )

## 📈 Logistic Regression — Threshold 0.5

| Metric | Score |
|---|---:|
| **Accuracy** | **93.71%** |
| **Precision** | **59.84%** |
| **Recall** | **15.10%** |
| **F1 Score** | **24.11%** |
| **ROC-AUC** | **82.50%** |

### Decision Tree

In [44]:
from sklearn.tree import DecisionTreeClassifier

In [45]:
dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train_final , y_train)

y_pred_dt = dt.predict(X_test_final)
y_prob_dt = dt.predict_proba(X_test_final)[:, 1]


print("Accuracy :", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall   :", recall_score(y_test, y_pred_dt))
print("F1 Score :", f1_score(y_test, y_pred_dt))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_dt))

Accuracy : 0.8989397396322641
Precision: 0.2590529247910863
Recall   : 0.2826747720364742
F1 Score : 0.2703488372093023
ROC-AUC  : 0.6128505097347079


## 🌳 Decision Tree — Untuned

| Metric | Score |
|---|---:|
| **Accuracy** | **89.89%** |
| **Precision** | **25.91%** |
| **Recall** | **28.27%** |
| **F1 Score** | **27.03%** |
| **ROC-AUC** | **61.29%** |

### Random Forest

In [46]:
from sklearn.ensemble import RandomForestClassifier

In [48]:
rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train_final , y_train)

y_pred_rf = rf.predict(X_test_final)
y_prob_rf = rf.predict_proba(X_test_final)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf))

Accuracy : 0.9350758287478191
Precision: 0.5277382645803699
Recall   : 0.1879432624113475
F1 Score : 0.2771759432200224
ROC-AUC  : 0.8310675959591105


In [50]:
baseline_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        0.937055,
        0.898940,
        0.935076
    ],
    "Precision": [
        0.598394,
        0.259053,
        0.527738
    ],
    "Recall": [
        0.150963,
        0.282675,
        0.187943
    ],
    "F1 Score": [
        0.241100,
        0.270349,
        0.277176
    ],
    "ROC-AUC": [
        0.824983,
        0.612851,
        0.831068
    ]
})

baseline_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.937055,0.598394,0.150963,0.241100,0.824983
1,Decision Tree,0.898940,0.259053,0.282675,0.270349,0.612851
2,Random Forest,0.935076,0.527738,0.187943,0.277176,0.831068


In [52]:
baseline_results.to_csv(
    "../reports/baseline_results.csv",
    index=False
)

## Baseline Model Comparison

The three baseline models show different performance characteristics.

- Logistic Regression achieves the highest accuracy and precision.
- Decision Tree achieves the highest recall among the three baseline models.
- Random Forest achieves the highest F1 Score and ROC-AUC.
- The relatively low recall across all models at the default threshold of 0.5 indicates that many actual defaulters are not being identified.
- Since the dataset is highly imbalanced, accuracy alone is not sufficient for evaluating model performance.
- Further improvement will therefore focus on tree-based models and their hyperparameters.